In [1]:
!rm -rf /kaggle/working/visual_search
!git clone https://github.com/Nihit3003/Visual-Product-Search-Engine /kaggle/working/visual_search

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import torchvision
import torch
print(np.__version__)
print("Torchvision OK")
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import subprocess
import sys
packages = [
    "hnswlib",
    "open-clip-torch==2.24.0",
    "transformers==4.41.2",
    "accelerate==0.28.0",
    "sentencepiece",
    "ultralytics==8.3.0",
    "timm",
    "einops",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("Packages installed")

Cloning into '/kaggle/working/visual_search'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 396 (delta 156), reused 51 (delta 49), pack-reused 169 (from 2)
Receiving objects: 100% (396/396), 214.66 KiB | 13.42 MiB/s, done.
Resolving deltas: 100% (217/217), done.
2.0.2
Torchvision OK
CUDA Available: True
GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatib

Packages installed


In [2]:
import os
import sys
PROJECT_ROOT = "/kaggle/working/visual_search"
sys.path.insert(0, PROJECT_ROOT)

# =========================================================
# DATASETS & PATHS
# =========================================================
DATASET_ROOT = "/kaggle/input/datasets/abhinavkishan123/deepfashion-inshop-dataset/Dataset"
CAPTION_DATASET = "/kaggle/input/captions"
CAPTIONS_PATH = f"{CAPTION_DATASET}/captions.json"

# CORRECTED CHECKPOINT PATH
CKPT_PATH = "/kaggle/input/datasets/nandamitra/clip-finetuned-best/clip_finetuned_best_576.pt"

INDEX_DIR = "/kaggle/working/index"
os.makedirs(INDEX_DIR, exist_ok=True)

print("Checkpoint:", CKPT_PATH)
print("Captions:", CAPTIONS_PATH)

Checkpoint: /kaggle/input/datasets/nandamitra/clip-finetuned-best/clip_finetuned_best_576.pt
Captions: /kaggle/input/captions/captions.json


In [3]:
# 1. Fix __init__.py
init_content = """# Visual Product Search Engine — Source Package
from .dataset import (build_dataloaders, get_clip_transform, parse_eval_partition, parse_bboxes, infer_category, DeepFashionDataset, bbox_crop)
from .model import (VisualSearchModel, SupConLoss)
from .blip_module import (FashionCaptioner, ITMReranker)
from .localizer import (YOLOLocalizer)
from .index import (HNSWIndex)
from .metrics import (evaluate, MetricResults, evaluate_multi_seed)
"""
with open("/kaggle/working/visual_search/src/__init__.py", "w") as f:
    f.write(init_content)
print("Fixed __init__.py")

# 2. Fix blip_module.py
blip_content = """from __future__ import annotations
import torch
from PIL import Image
from transformers import (Blip2Processor, Blip2ForConditionalGeneration, BlipImageProcessor, AutoTokenizer)
from transformers import (BlipForImageTextRetrieval, BlipProcessor)

class FashionCaptioner:
    PROMPT = "Describe the clothing item including color, material, fit, texture, style, and garment type."
    def __init__(self, model_name: str = "Salesforce/blip2-flan-t5-xl", device: str = "cuda", use_fp16: bool = True):
        self.device = device
        dtype = torch.float16 if (use_fp16 and device == "cuda") else torch.float32
        print(f"[BLIP-2] Loading captioner: {model_name}")
        image_processor = BlipImageProcessor.from_pretrained(model_name)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.processor = Blip2Processor(image_processor=image_processor, tokenizer=tokenizer)
        self.model = Blip2ForConditionalGeneration.from_pretrained(model_name, torch_dtype=dtype).to(device)
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        print("[BLIP-2] Captioner ready.")

    @torch.no_grad()
    def caption(self, images: list[Image.Image] | Image.Image, batch_size: int = 4) -> list[str]:
        if isinstance(images, Image.Image): images = [images]
        outputs = []
        for i in range(0, len(images), batch_size):
            batch = images[i : i + batch_size]
            try:
                inputs = self.processor(images=batch, text=[self.PROMPT] * len(batch), return_tensors="pt", padding=True).to(self.device)
                generated = self.model.generate(**inputs, max_new_tokens=40, num_beams=3)
                decoded = self.processor.batch_decode(generated, skip_special_tokens=True)
                outputs.extend([d.strip() for d in decoded])
            except Exception as e:
                print(f"[BLIP-2] Batch failed: {e}")
                outputs.extend([""] * len(batch))
        return outputs

class ITMReranker:
    def __init__(self, model_name: str = "Salesforce/blip-itm-base-coco", device: str = "cuda", use_fp16: bool = True):
        self.device = device
        dtype = torch.float16 if (use_fp16 and device == "cuda") else torch.float32
        print(f"[BLIP-ITM] Loading {model_name}")
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForImageTextRetrieval.from_pretrained(model_name, torch_dtype=dtype).to(device)
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        print("[BLIP-ITM] Ready.")

    @torch.no_grad()
    def score(self, query_image: Image.Image, captions: list[str], batch_size: int = 8) -> list[float]:
        scores = []
        for i in range(0, len(captions), batch_size):
            caps = captions[i : i + batch_size]
            try:
                inputs = self.processor(images=[query_image] * len(caps), text=caps, return_tensors="pt", padding=True, truncation=True).to(self.device)
                out = self.model(**inputs, use_itm_head=True)
                probs = out.itm_score.softmax(dim=-1)[:, 1]
                scores.extend(probs.float().cpu().tolist())
            except Exception as e:
                print(f"[BLIP-ITM] Batch failed: {e}")
                scores.extend([0.0] * len(caps))
        return scores

    def rerank(self, query_image: Image.Image, candidates: list[dict], caption_key: str = "caption") -> list[dict]:
        captions = [c.get(caption_key, "") for c in candidates]
        scores = self.score(query_image, captions)
        for cand, sc in zip(candidates, scores): cand["itm_score"] = sc
        return sorted(candidates, key=lambda x: x["itm_score"], reverse=True)
"""
with open("/kaggle/working/visual_search/src/blip_module.py", "w") as f:
    f.write(blip_content)
print("Updated blip_module.py successfully.")

Fixed __init__.py
Updated blip_module.py successfully.


In [4]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --captions_json "{CAPTIONS_PATH}" \
    --index_dir "{INDEX_DIR}" \
    --condition A \
    --alpha 1.0 \
    --batch_size 64 \
    --embed_dim 768 \
    --unfreeze_last_n 0 \
    --use_gt_bbox

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-05-13 11:45:01.207430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778672701.625497     134 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778672701.747659     134 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778672702.810461     134 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mor

In [5]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --captions_json "{CAPTIONS_PATH}" \
    --index_dir "{INDEX_DIR}" \
    --condition B \
    --alpha 0.7 \
    --batch_size 64 \
    --embed_dim 768 \
    --unfreeze_last_n 0 \
    --use_gt_bbox

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-05-13 12:18:17.416220: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778674697.439490     941 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778674697.447032     941 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778674697.466788     941 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mor

In [6]:
!python /kaggle/working/visual_search/scripts/build_index.py \
    --dataset_root "{DATASET_ROOT}" \
    --captions_json "{CAPTIONS_PATH}" \
    --ckpt_path "{CKPT_PATH}" \
    --index_dir "{INDEX_DIR}" \
    --condition C \
    --alpha 0.7 \
    --batch_size 48 \
    --embed_dim 768 \
    --use_gt_bbox

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-05-13 14:38:08.822522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778683088.846272    2271 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778683088.854289    2271 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778683088.875498    2271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mor

In [7]:
import os
important = [
    f"{INDEX_DIR}/condition_A_alpha1.0/hnsw.bin",
    f"{INDEX_DIR}/condition_B_alpha0.7/hnsw.bin",
    f"{INDEX_DIR}/condition_C_alpha0.7/hnsw.bin",
]
print("\nGenerated indexes:\n")
for p in important:
    print("✓" if os.path.exists(p) else "✗", p)


Generated indexes:

✓ /kaggle/working/index/condition_A_alpha1.0/hnsw.bin
✓ /kaggle/working/index/condition_B_alpha0.7/hnsw.bin
✓ /kaggle/working/index/condition_C_alpha0.7/hnsw.bin
